In [1]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-chroma chromadb fastembed pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/6

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pypdf

print("PDF libraries are working")

/tmp/ipykernel_1340/4081979253.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PDF libraries are working


In [7]:
from google.colab import files

uploaded = files.upload()

Saving type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf to type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf
Saving type-2-diabetes-in-adults-management-pdf-1837338615493.pdf to type-2-diabetes-in-adults-management-pdf-1837338615493.pdf


In [11]:
PDF_PATHS = list(uploaded.keys())

print("Number of PDFs:", len(PDF_PATHS))

for pdf in PDF_PATHS:
    print(pdf)

Number of PDFs: 2
type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf
type-2-diabetes-in-adults-management-pdf-1837338615493.pdf


In [12]:
from langchain_community.document_loaders import PyPDFLoader

all_pages = []

for pdf_path in PDF_PATHS:
    print("Loading:", pdf_path)

    loader = PyPDFLoader(pdf_path)
    pages = loader.load()

    print("Pages loaded:", len(pages))

    all_pages.extend(pages)

print("Total pages loaded:", len(all_pages))

Loading: type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf
Pages loaded: 64
Loading: type-2-diabetes-in-adults-management-pdf-1837338615493.pdf
Pages loaded: 131
Total pages loaded: 195


In [13]:
print(all_pages[0].page_content[:2000])

Type 1 diabetes in adults: 
diagnosis and management 
NICE guideline 
Published: 26 August 2015 
Last updated: 17 August 2022 
www.nice.org.uk/guidance/ng17 
© NICE 2026. All rights reserved. Subject to Notice of rights (https://www.nice.org.uk/terms-and-
conditions#notice-of-rights).


In [14]:
print(all_pages[64].page_content[:2000])

Type 2 diabetes in adults: 
management 
NICE guideline 
Published: 2 December 2015 
Last updated: 18 February 2026 
www.nice.org.uk/guidance/ng28 
© NICE 2026. All rights reserved. Subject to Notice of rights (https://www.nice.org.uk/terms-and-
conditions#notice-of-rights).


In [15]:
import os

for page in all_pages:
    page.metadata["document_name"] = os.path.basename(
        page.metadata["source"]
    )

    page.metadata["page_number"] = (
        page.metadata.get("page", 0) + 1
    )

In [16]:
print(all_pages[0].metadata)

{'producer': 'Prince 12.5 (www.princexml.com)', 'creator': 'NICE Publications', 'creationdate': '2022-08-17T00:00:00+00:00', 'keywords': 'NG17', 'subject': 'Type 1 diabetes in adults: diagnosis and management (NG17)', 'author': 'National Institute for Health and Care Excellence (NICE)', 'title': 'Type 1 diabetes in adults: diagnosis and management', 'source': 'type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf', 'total_pages': 64, 'page': 0, 'page_label': '1', 'document_name': 'type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf', 'page_number': 1}


In [17]:
print("Document:", all_pages[0].metadata["document_name"])
print("Page number:", all_pages[0].metadata["page_number"])

Document: type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf
Page number: 1


In [18]:
print(all_pages[10].page_content[:2000])

approach. [2004, amended 2021] 
1.2.3 Provide adults with type 1 diabetes with: 
• access to services by different methods (including phone and email) during 
working hours 
• information about out-of-hours services staffed by people with diabetes 
expertise. [2004] 
1.2.4 View each adult with type 1 diabetes as an individual, rather than as a member of 
any cultural, economic or health-affected group (also see recommendations 1.4.5 
and 1.4.14 on cultural preferences in the section on dietary advice). [2004, 
amended 2015] 
1.2.5 Jointly agree an individual care plan with the adult with type 1 diabetes. Review 
this plan annually and amend it as needed, taking into account changes in the 
person's wishes, circumstances and medical findings. [2004, amended 2015] 
1.2.6 Individual care plans should include: 
• diabetes education, including dietary advice (see the sections on education 
and information and dietary management) 
• insulin therapy, including dosage adjustment (see the secti

In [19]:
import re

def clean_text(text):
    # Replace non-breaking spaces with normal spaces
    text = text.replace("\u00a0", " ")

    # Replace repeated spaces/tabs with one space
    text = re.sub(r"[ \t]+", " ", text)

    # Remove spaces at the beginning of new lines
    text = re.sub(r"\n[ \t]+", "\n", text)

    # If there are more than 2 empty lines, keep only 2
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [20]:
sample_raw = all_pages[10].page_content
sample_clean = clean_text(sample_raw)

print("===== BEFORE CLEANING =====")
print(sample_raw[:1500])

print("\n\n===== AFTER CLEANING =====")
print(sample_clean[:1500])

===== BEFORE CLEANING =====
approach. [2004, amended 2021] 
1.2.3 Provide adults with type 1 diabetes with: 
• access to services by different methods (including phone and email) during 
working hours 
• information about out-of-hours services staffed by people with diabetes 
expertise. [2004] 
1.2.4 View each adult with type 1 diabetes as an individual, rather than as a member of 
any cultural, economic or health-affected group (also see recommendations 1.4.5 
and 1.4.14 on cultural preferences in the section on dietary advice). [2004, 
amended 2015] 
1.2.5 Jointly agree an individual care plan with the adult with type 1 diabetes. Review 
this plan annually and amend it as needed, taking into account changes in the 
person's wishes, circumstances and medical findings. [2004, amended 2015] 
1.2.6 Individual care plans should include: 
• diabetes education, including dietary advice (see the sections on education 
and information and dietary management) 
• insulin therapy, including dosa

In [21]:
for page in all_pages:
    page.page_content = clean_text(page.page_content)

print("Cleaning completed for all pages ✅")
print("Total cleaned pages:", len(all_pages))

Cleaning completed for all pages ✅
Total cleaned pages: 195


In [22]:
print("===== NOUR'S INGESTION SUMMARY =====")

print("Total PDFs:", len(PDF_PATHS))
print("Total pages:", len(all_pages))

print("\nSample metadata:")
print("Document:", all_pages[0].metadata["document_name"])
print("Page:", all_pages[0].metadata["page_number"])

print("\nSample cleaned text:")
print(all_pages[10].page_content[:1000])

===== NOUR'S INGESTION SUMMARY =====
Total PDFs: 2
Total pages: 195

Sample metadata:
Document: type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf
Page: 1

Sample cleaned text:
approach. [2004, amended 2021] 
1.2.3 Provide adults with type 1 diabetes with: 
• access to services by different methods (including phone and email) during 
working hours 
• information about out-of-hours services staffed by people with diabetes 
expertise. [2004] 
1.2.4 View each adult with type 1 diabetes as an individual, rather than as a member of 
any cultural, economic or health-affected group (also see recommendations 1.4.5 
and 1.4.14 on cultural preferences in the section on dietary advice). [2004, 
amended 2015] 
1.2.5 Jointly agree an individual care plan with the adult with type 1 diabetes. Review 
this plan annually and amend it as needed, taking into account changes in the 
person's wishes, circumstances and medical findings. [2004, amended 2015] 
1.2.6 Individual care plans

In [23]:
print("Total pages:", len(all_pages))

print("\nFirst page metadata:")
print(all_pages[0].metadata)

print("\nFirst page text:")
print(all_pages[0].page_content[:2000])

Total pages: 195

First page metadata:
{'producer': 'Prince 12.5 (www.princexml.com)', 'creator': 'NICE Publications', 'creationdate': '2022-08-17T00:00:00+00:00', 'keywords': 'NG17', 'subject': 'Type 1 diabetes in adults: diagnosis and management (NG17)', 'author': 'National Institute for Health and Care Excellence (NICE)', 'title': 'Type 1 diabetes in adults: diagnosis and management', 'source': 'type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf', 'total_pages': 64, 'page': 0, 'page_label': '1', 'document_name': 'type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf', 'page_number': 1}

First page text:
Type 1 diabetes in adults: 
diagnosis and management 
NICE guideline 
Published: 26 August 2015 
Last updated: 17 August 2022 
www.nice.org.uk/guidance/ng17 
© NICE 2026. All rights reserved. Subject to Notice of rights (https://www.nice.org.uk/terms-and-
conditions#notice-of-rights).


In [24]:
for i, page in enumerate(all_pages[:3]):

    print("=" * 80)
    print("PAGE:", page.metadata["page_number"])
    print("DOCUMENT:", page.metadata["document_name"])
    print("=" * 80)

    print(page.page_content[:1500])
    print()

PAGE: 1
DOCUMENT: type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf
Type 1 diabetes in adults: 
diagnosis and management 
NICE guideline 
Published: 26 August 2015 
Last updated: 17 August 2022 
www.nice.org.uk/guidance/ng17 
© NICE 2026. All rights reserved. Subject to Notice of rights (https://www.nice.org.uk/terms-and-
conditions#notice-of-rights).

PAGE: 2
DOCUMENT: type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf
Your responsibility 
The recommendations in this guideline represent the view of NICE, arrived at after careful 
consideration of the evidence available. When exercising their judgement, professionals 
and practitioners are expected to take this guideline fully into account, alongside the 
individual needs, preferences and values of their patients or the people using their service. 
It is not mandatory to apply the recommendations, and the guideline does not override the 
responsibility to make decisions appropriate to the ci

In [25]:
for page in all_pages[:20]:
    for line in page.page_content.splitlines():
        line = line.strip()
        if re.match(r"^\d+(\.\d+)*\s+.+", line):
            print(line)

1.1 Diagnosis and early care plan ......................................................................................................... 6
1.2 Support and individualised care .................................................................................................... 10
1.3 Education and information .............................................................................................................. 12
1.4 Dietary management ....................................................................................................................... 13
1.5 Physical activity ............................................................................................................................... 16
1.6 Blood glucose management ........................................................................................................... 17
1.7 Insulin therapy ...................................................................................................................

In [26]:
def is_section_heading(line):
    line = line.strip()

    if not line:
        return False

    pattern = r"^\d+\.\d+\s+.+"

    return bool(re.match(pattern, line))

In [27]:
test_lines = [
    "1.1 Diagnosis and early care plan",
    "1.2 Support and individualised care",
    "1.10 Ketone monitoring and managing diabetic ketoacidosis",
    "1.1.1 Make an initial diagnosis of type 1 diabetes on clinical grounds in adults",
    "1.2.3 Provide adults with type 1 diabetes with:",
    "People with diabetes should..."
]

for line in test_lines:
    print(is_section_heading(line), "→", line)

True → 1.1 Diagnosis and early care plan
True → 1.2 Support and individualised care
True → 1.10 Ketone monitoring and managing diabetic ketoacidosis
False → 1.1.1 Make an initial diagnosis of type 1 diabetes on clinical grounds in adults
False → 1.2.3 Provide adults with type 1 diabetes with:
False → People with diabetes should...


In [28]:
def group_pages_by_section(pages):
    grouped_sections = []
    current_section = "Unknown"
    current_document = None
    current_text = []
    current_pages = []

    for page in pages:
        document_name = page.metadata["document_name"]
        page_number = page.metadata["page_number"]
        for line in page.page_content.splitlines():
            line = line.strip()
            if not line:
                continue

            # If this line is a new main section
            if is_section_heading(line):
                # Save the previous section first
                if current_text:
                    grouped_sections.append({
                        "document_name": current_document,
                        "section": current_section,
                        "text": "\n".join(current_text),
                        "page_numbers": current_pages.copy()
                    })

                # Start the new section
                current_section = line
                current_document = document_name
                current_text = []
                current_pages = [page_number]

            else:
                # Normal text or recommendation
                current_text.append(line)
                # Keep track of pages belonging to this section
                if page_number not in current_pages:
                    current_pages.append(page_number)

    # Save the last section
    if current_text:

        grouped_sections.append({
            "document_name": current_document,
            "section": current_section,
            "text": "\n".join(current_text),
            "page_numbers": current_pages.copy()
        })

    return grouped_sections

In [29]:
grouped_sections = group_pages_by_section(all_pages)
print("Number of grouped sections:", len(grouped_sections))

Number of grouped sections: 73


In [30]:
for i, page in enumerate(all_pages):

    if page.metadata["document_name"] != all_pages[0].metadata["document_name"]:

        print("Second PDF starts at page index:", i)
        print("Document:", page.metadata["document_name"])
        print("Page number:", page.metadata["page_number"])
        print(page.page_content[:2000])

        break

Second PDF starts at page index: 64
Document: type-2-diabetes-in-adults-management-pdf-1837338615493.pdf
Page number: 1
Type 2 diabetes in adults: 
management 
NICE guideline 
Published: 2 December 2015 
Last updated: 18 February 2026 
www.nice.org.uk/guidance/ng28 
© NICE 2026. All rights reserved. Subject to Notice of rights (https://www.nice.org.uk/terms-and-
conditions#notice-of-rights).


In [31]:
second_pdf_name = all_pages[64].metadata["document_name"]

for page in all_pages:

    if page.metadata["document_name"] != second_pdf_name:
        continue

    for line in page.page_content.splitlines():

        line = line.strip()

        if is_section_heading(line):

            print("First detected section:")
            print(line)

            print("Page number:", page.metadata["page_number"])

            break

    else:
        continue

    break

First detected section:
1.1 Tailoring care to a person's needs ................................................................................................. 7
Page number: 3


In [32]:
second_pdf_name = all_pages[64].metadata["document_name"]

for page in all_pages:

    if (
        page.metadata["document_name"] == second_pdf_name
        and page.metadata["page_number"] == 7
    ):

        print("Document:", page.metadata["document_name"])
        print("Page:", page.metadata["page_number"])
        print("=" * 80)
        print(page.page_content[:3000])

        break

Document: type-2-diabetes-in-adults-management-pdf-1837338615493.pdf
Page: 7
Individualised care 
1.1 Tailoring care to a person's needs 
1.1.1 Adopt an individualised approach to diabetes care that is tailored to the needs 
and circumstances of adults with type 2 diabetes, taking into account their 
personal preferences, comorbidities and risks from polypharmacy, and their 
likelihood of benefiting from long-term interventions. Such an approach is 
especially important in the context of multimorbidity. See also NICE's guidelines 
on assessing and managing multimorbidity and on medicines optimisation. [2015, 
amended 2026] 
1.1.2 Reassess the person's needs and circumstances at each review and think about 
whether to stop any medicines that are not effective. [2015] 
1.1.3 Take into account any disabilities, including visual impairment, when planning and 
delivering care for adults with type 2 diabetes. It is particularly important to 
choose the technology that best supports a person'

In [33]:
def group_pages_by_section(pages):

    grouped_sections = []

    current_section = None
    current_document = None
    current_pages = []

    # First real content page for each PDF
    first_content_page = {
        "type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701 (2).pdf": 6,
        "type-2-diabetes-in-adults-management-pdf-1837338615493 (1).pdf": 7
    }

    for page in pages:

        document_name = page.metadata["document_name"]
        page_number = page.metadata["page_number"]

        # Skip cover / front matter / contents
        if page_number < first_content_page.get(document_name, 1):
            continue

        # Process lines inside the page
        lines = page.page_content.splitlines()

        for line in lines:

            line = line.strip()

            if not line:
                continue

            # -----------------------------
            # New section detected
            # -----------------------------
            if is_section_heading(line):

                # Save previous section
                if current_section is not None:

                    grouped_sections.append({
                        "document_name": current_document,
                        "section": current_section,
                        "pages": current_pages.copy()
                    })

                # Start new section
                current_section = line
                current_document = document_name
                current_pages = []

            else:

                # Add text to current page
                if current_section is not None:

                    # Check if this page already exists
                    if not current_pages or current_pages[-1]["page_number"] != page_number:

                        current_pages.append({
                            "page_number": page_number,
                            "text": line
                        })

                    else:

                        current_pages[-1]["text"] += "\n" + line

    # Save final section
    if current_section is not None:

        grouped_sections.append({
            "document_name": current_document,
            "section": current_section,
            "pages": current_pages.copy()
        })

    return grouped_sections

In [34]:
grouped_sections = group_pages_by_section(all_pages)

print("Number of grouped sections:", len(grouped_sections))

Number of grouped sections: 118


In [35]:
first_section = grouped_sections[0]

print("Document:")
print(first_section.get("document_name", "No document name"))

print("\nSection:")
print(first_section.get("section", "No section"))

pages = first_section.get("pages", [])
print(f"\nNumber of pages: {len(pages)}")

if pages:
    first_page = pages[0]
    print("\nFirst page structure:")
    print(first_page)
else:
    print("\nNo pages found in this section.")

Document:
type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf

Section:
1.1 Diagnosis and early care plan ......................................................................................................... 6

Number of pages: 0

No pages found in this section.


In [36]:
for page in first_section["pages"]:

    print("=" * 80)
    print("PAGE:", page["page_number"])
    print("=" * 80)
    print(page["text"][:300])

In [37]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=850,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

print("Chunk splitter is ready")

Chunk splitter is ready


In [38]:
# def create_chunks_from_section(
#     section,
#     splitter,
#     chunk_id_start=1
# ):

#     chunks = []
#     chunk_counter = chunk_id_start

#     for page in section["pages"]:

#         page_number = page["page_number"]
#         page_text = page["text"]

#         page_chunks = splitter.split_text(page_text)

#         for chunk_text in page_chunks:

#             chunk = Document(
#                 page_content=chunk_text,
#                 metadata={
#                     "document_name": section["document_name"],
#                     "section": section["section"],
#                     "page_number": page_number,
#                     "chunk_id": f"chunk_{chunk_counter:04d}"
#                 }
#             )

#             chunks.append(chunk)

#             chunk_counter += 1

#     return chunks, chunk_counter

In [39]:
from langchain_core.documents import Document

def create_all_chunks(grouped_sections, splitter):

    all_chunks = []
    chunk_counter = 1

    for section in grouped_sections:

        # Combine all pages belonging to this section
        section_text = ""
        page_boundaries = []

        for page in section["pages"]:

            start_position = len(section_text)

            section_text += page["text"] + "\n"

            end_position = len(section_text)

            page_boundaries.append({
                "page_number": page["page_number"],
                "start": start_position,
                "end": end_position
            })

        # Split the whole SECTION
        section_chunks = splitter.split_text(section_text)

        # Track where each chunk came from
        search_start = 0

        for chunk_text in section_chunks:

            # Find chunk position inside original section text
            chunk_start = section_text.find(
                chunk_text,
                search_start
            )

            if chunk_start == -1:
                chunk_start = search_start

            chunk_end = chunk_start + len(chunk_text)

            # Find pages covered by this chunk
            chunk_pages = []

            for boundary in page_boundaries:

                if (
                    boundary["end"] > chunk_start
                    and boundary["start"] < chunk_end
                ):
                    chunk_pages.append(
                        boundary["page_number"]
                    )

            # Fallback
            if not chunk_pages:
                chunk_pages = [page_boundaries[0]["page_number"]]

            chunk = Document(
                page_content=chunk_text,
                metadata={
                    "document_name": section["document_name"],
                    "section": section["section"],
                    "page_number": chunk_pages[0],
                    "page_numbers": chunk_pages,
                    "chunk_id": f"chunk_{chunk_counter:04d}"
                }
            )

            all_chunks.append(chunk)

            chunk_counter += 1

            # Move search position forward
            search_start = chunk_start + 1

    return all_chunks

In [40]:
{
    "document_name": "...",
    "section": "1.1 Diagnosis and early care plan",
    "page_number": 6,
    "page_numbers": [6, 7],
    "chunk_id": "chunk_0001"
}

{'document_name': '...',
 'section': '1.1 Diagnosis and early care plan',
 'page_number': 6,
 'page_numbers': [6, 7],
 'chunk_id': 'chunk_0001'}

In [41]:
chunks = create_all_chunks(
    grouped_sections,
    splitter
)

print("Total chunks:", len(chunks))

Total chunks: 462


In [42]:
for i, chunk in enumerate(chunks[:3], 1):

    print("=" * 80)
    print(f"CHUNK {i}")
    print("=" * 80)

    print("\nTEXT:")
    print(chunk.page_content)

    print("\nTEXT LENGTH:")
    print(len(chunk.page_content))

    print("\nMETADATA:")
    print(chunk.metadata)

CHUNK 1

TEXT:
Terms used in this guideline ................................................................................................................. 48
Recommendations for research .................................................................................................49
1 Clinical features for distinguishing between type 1 diabetes and other types of diabetes ........ 49
2 The use of C-peptide in diagnosing diabetes ................................................................................. 49
3 Use of routinely collected real-world data to examine the effectiveness and cost
effectiveness of continuous glucose monitoring ............................................................................... 50

TEXT LENGTH:
721

METADATA:
{'document_name': 'type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf', 'section': '1.14 Managing complications ....................................................................................................

In [43]:
required_fields = [
    "document_name",
    "section",
    "page_number",
    "chunk_id"
]

for i, chunk in enumerate(chunks):

    for field in required_fields:

        assert field in chunk.metadata, (
            f"Missing '{field}' "
            f"in chunk {i}"
        )

print("All chunks contain required metadata")

All chunks contain required metadata


In [44]:
chunk_ids = [
    chunk.metadata["chunk_id"]
    for chunk in chunks
]

print("Total chunks:", len(chunk_ids))
print("Unique chunk IDs:", len(set(chunk_ids)))

assert len(chunk_ids) == len(set(chunk_ids))

print("All chunk IDs are unique")

Total chunks: 462
Unique chunk IDs: 462
All chunk IDs are unique


In [45]:
from collections import Counter

document_counts = Counter(
    chunk.metadata["document_name"]
    for chunk in chunks
)

print("Chunks per document:")

for document, count in document_counts.items():
    print(document)
    print("Chunks:", count)
    print()

Chunks per document:
type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf
Chunks: 157

type-2-diabetes-in-adults-management-pdf-1837338615493.pdf
Chunks: 305



In [46]:
sections = set(
    chunk.metadata["section"]
    for chunk in chunks
)

print("Number of unique sections:", len(sections))

Number of unique sections: 72


## Mariam 2 – Embeddings Generation & Chroma Vector Store Creation

In [47]:
import os
import shutil
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")

cleaned_chunks = []
for doc in chunks:
    clean_meta = {}
    for k, v in doc.metadata.items():
        if v is None:
            clean_meta[k] = ""
        elif isinstance(v, (str, int, float, bool)):
            clean_meta[k] = v
        else:
            clean_meta[k] = str(v)
    doc.metadata = clean_meta
    cleaned_chunks.append(doc)

persist_directory = "./chroma_db"
if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)

vector_db = Chroma.from_documents(
    documents=cleaned_chunks,
    embedding=embedding_model,
    persist_directory=persist_directory,
    collection_name="diabetes_educational_rag"
)

print("success")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

success


In [48]:
stored_count = vector_db._collection.count()
input_count = len(chunks)

print("Chroma DB Indexing Status:")
print(f"- Total input chunks: {input_count}")
print(f"- Total vectors stored: {stored_count}")

if stored_count == input_count:
    print(f"All {stored_count} chunks were indexed and stored successfully.")
else:
    print(f"Warning: Count mismatch (expected {input_count}, but found {stored_count} in database).")

Chroma DB Indexing Status:
- Total input chunks: 462
- Total vectors stored: 462
All 462 chunks were indexed and stored successfully.


## Laila Part - Retreival + Test

In [49]:
def retrieve_with_similarity(question, k=4):
    """
    Takes a user question, embeds it, runs similarity search on Chroma,
    and returns the Top-K (chunk, similarity_score) pairs, ranked highest first.
    """
    return vector_db.similarity_search_with_relevance_scores(question, k=k)


def print_retrieval_results(question, k=4):
    results = retrieve_with_similarity(question, k=k)

    print(f"QUESTION: {question}\n")
    for rank, (doc, score) in enumerate(results, start=1):
        print(f"Rank {rank}")
        print("  Document :", doc.metadata.get("document_name"))
        print("  Page     :", doc.metadata.get("page_number"))
        print("  Section  :", doc.metadata.get("section"))
        print("  Chunk ID :", doc.metadata.get("chunk_id"))
        print("  Score    :", round(score, 4))
        print("  Text     :", doc.page_content[:200].replace("\n", " "), "...")
        print()

    return results

In [50]:
# First simple, clear test question
first_results = print_retrieval_results("What are the diagnostic criteria for diabetes?", k=4)

QUESTION: What are the diagnostic criteria for diabetes?

Rank 1
  Document : type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf
  Page     : 6
  Section  : 1.1 Diagnosis and early care plan
  Chunk ID : chunk_0007
  Score    : 0.664
  Text     : Initial diagnosis 1.1.1 Make an initial diagnosis of type 1 diabetes on clinical grounds in adults presenting with hyperglycaemia. Bear in mind that people with type 1 diabetes typically (but not alwa ...

Rank 2
  Document : type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf
  Page     : 7
  Section  : 1.1 Diagnosis and early care plan
  Chunk ID : chunk_0008
  Score    : 0.6457
  Text     : [2022] 1.1.3 Take into consideration the possibility of other diabetes subtypes and revisit the diagnosis at subsequent clinical reviews. Carry out further investigations if there is uncertainty (see  ...

Rank 3
  Document : type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf
  Page     : 5

### Load Eman's test questions directly from `RAG_Test_Questions.pdf`

In [51]:
QUESTIONS_FILENAME = "RAG_Test_Questions.pdf"

if os.path.exists(QUESTIONS_FILENAME):
    QUESTIONS_PDF_PATH = QUESTIONS_FILENAME
    print("Found existing file in /content, using it:", QUESTIONS_PDF_PATH)
else:
    from google.colab import files
    print("Upload RAG_Test_Questions.pdf")
    uploaded_questions = files.upload()
    QUESTIONS_PDF_PATH = list(uploaded_questions.keys())[0]
    print("Loaded:", QUESTIONS_PDF_PATH)

Found existing file in /content, using it: RAG_Test_Questions.pdf


In [52]:
from pypdf import PdfReader

reader = PdfReader(QUESTIONS_PDF_PATH)
raw_text = "\n".join(page.extract_text() for page in reader.pages)
lines = [l.strip() for l in raw_text.split("\n") if l.strip()]

SKIP_LINES = ("#", "Question", "Source PDF", "Page", "Why unsupported")
SKIP_PREFIXES = (
    "Everyday-style", "Testing note", "right page", "answer isn't",
    "Simple, plain", "(Type 2", "RAG Test",
)

supported_questions = []
unsupported_questions = []
mode = None
i = 0

while i < len(lines):
    line = lines[i]

    if line.startswith("Part 1"):
        mode = "supported"; i += 1; continue
    if line.startswith("Part 2"):
        mode = "unsupported"; i += 1; continue
    if line in SKIP_LINES or line.startswith(SKIP_PREFIXES):
        i += 1; continue

    # ---- Part 1: # | Question (1-2 lines) | Source PDF | Page ----
    if mode == "supported" and re.fullmatch(r"\d{1,2}", line):
        idx = int(line); i += 1
        q_parts = []
        while i < len(lines) and not re.fullmatch(r"type-[12]\.pdf", lines[i]):
            q_parts.append(lines[i]); i += 1
        question = " ".join(q_parts)
        source = lines[i] if i < len(lines) else None; i += 1
        page = lines[i] if i < len(lines) and lines[i].isdigit() else None; i += 1
        supported_questions.append({
            "number": idx,
            "question": question,
            "expected_source": source,
            "expected_page": int(page) if page else None,
        })
        continue

    # ---- Part 2: # | Question (ends with "?") | Why unsupported (rest) ----
    if mode == "unsupported" and re.fullmatch(r"\d{1,2}", line):
        idx = int(line); i += 1
        q_parts = []
        while i < len(lines) and not lines[i].endswith("?"):
            q_parts.append(lines[i]); i += 1
        if i < len(lines):
            q_parts.append(lines[i]); i += 1
        question = " ".join(q_parts)
        reason_parts = []
        while i < len(lines) and not (re.fullmatch(r"\d{1,2}", lines[i]) and int(lines[i]) == idx + 1) \
                and not lines[i].startswith("Testing note"):
            reason_parts.append(lines[i]); i += 1
        unsupported_questions.append({
            "number": idx,
            "question": question,
            "reason": " ".join(reason_parts),
        })
        continue

    i += 1

print("Parsed from RAG_Test_Questions.pdf ✅")
print("Supported questions  :", len(supported_questions))
print("Unsupported questions:", len(unsupported_questions))
print("\nExample supported  :", supported_questions[0])
print("Example unsupported:", unsupported_questions[0])

Parsed from RAG_Test_Questions.pdf ✅
Supported questions  : 15
Unsupported questions: 5

Example supported  : {'number': 1, 'question': 'What should my HbA1c number be if I have type 1 diabetes?', 'expected_source': 'type-1.pdf', 'expected_page': 18}
Example unsupported: {'number': 1, 'question': 'How much do diabetes medicines cost in Egypt?', 'reason': 'No pricing or local market info in either PDF'}


In [53]:
def _extract_pages(chunk_pages, fallback_page):
    """
    Mariam 2's cleaning step (cell-40) stringifies list metadata before storing
    in Chroma, so page_numbers comes back as "[3, 4]" (a string), not a list.
    Parse it back into ints; fall back to page_number if anything looks off.
    """
    if isinstance(chunk_pages, list):
        return [int(p) for p in chunk_pages if str(p).strip().lstrip("-").isdigit()]
    if isinstance(chunk_pages, str):
        found = re.findall(r"\d+", chunk_pages)
        if found:
            return [int(p) for p in found]
    return [fallback_page] if fallback_page is not None else []


def check_match(results, expected_source, expected_page, page_tolerance=3):
    """True if any retrieved chunk comes from the expected PDF and covers a nearby page."""
    expected_tag = "type-1" if "type-1" in expected_source else "type-2"
    for doc, _score in results:
        doc_name = doc.metadata.get("document_name", "").lower()
        chunk_pages = _extract_pages(
            doc.metadata.get("page_numbers"), doc.metadata.get("page_number")
        )
        if expected_tag in doc_name and any(
            abs(p - expected_page) <= page_tolerance for p in chunk_pages
        ):
            return True
    return False


print("===== PART 1: SUPPORTED QUESTIONS (from RAG_Test_Questions.pdf) =====\n")

supported_hits = 0
for item in supported_questions:
    results = print_retrieval_results(item["question"], k=4)
    matched = check_match(results, item["expected_source"], item["expected_page"])
    supported_hits += matched
    print(f"Expected: {item['expected_source']} p.{item['expected_page']}  ->  "
          f"{'✅ MATCH' if matched else '❌ NO MATCH'}")
    print("=" * 80)

print(f"\nSupported questions matched: {supported_hits}/{len(supported_questions)}")

===== PART 1: SUPPORTED QUESTIONS (from RAG_Test_Questions.pdf) =====

QUESTION: What should my HbA1c number be if I have type 1 diabetes?

Rank 1
  Document : type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf
  Page     : 17
  Section  : 1.6 Blood glucose management
  Chunk ID : chunk_0034
  Score    : 0.7166
  Text     : [2015] 1.6.5 If HbA1c monitoring is invalid because of disturbed erythrocyte turnover or abnormal haemoglobin type, estimate trends in blood glucose control using 1 of Type 1 diabetes in adults: diagn ...

Rank 2
  Document : type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf
  Page     : 17
  Section  : 1.6 Blood glucose management
  Chunk ID : chunk_0033
  Score    : 0.7158
  Text     : HbA1c measurement and targets Measurement 1.6.1 Measure HbA1c levels every 3 to 6 months in adults with type 1 diabetes. [2015] 1.6.2 Consider measuring HbA1c levels more often in adults with type 1 d ...

Rank 3
  Document : type-1-diabe

In [54]:
print("===== PART 2: UNSUPPORTED QUESTIONS (from RAG_Test_Questions.pdf) =====\n")
print("A correct system should show WEAK/irrelevant matches here — retrieval alone can't")
print("'refuse', that needs the LLM generation step (Day 2+), but low scores are the signal.\n")

for item in unsupported_questions:
    results = print_retrieval_results(item["question"], k=4)
    top_score = results[0][1] if results else 0
    print(f"Why unsupported: {item['reason']}")
    print(f"Top-1 similarity: {round(top_score, 4)}  ->  "
          f"{'⚠️ Suspiciously strong match, review this chunk' if top_score > 0.6 else '✅ Weak match, as expected'}")
    print("=" * 80)

===== PART 2: UNSUPPORTED QUESTIONS (from RAG_Test_Questions.pdf) =====

A correct system should show WEAK/irrelevant matches here — retrieval alone can't
'refuse', that needs the LLM generation step (Day 2+), but low scores are the signal.

QUESTION: How much do diabetes medicines cost in Egypt?

Rank 1
  Document : type-2-diabetes-in-adults-management-pdf-1837338615493.pdf
  Page     : 101
  Section  : 1.29 People living with obesity
  Chunk ID : chunk_0395
  Score    : 0.5409
  Text     : not reflect a significant change in current practice and are unlikely to increase resource use. There may be cost savings if people are no longer prescribed GLP-1 receptor agonists or tirzepatide and  ...

Rank 2
  Document : type-2-diabetes-in-adults-management-pdf-1837338615493.pdf
  Page     : 95
  Section  : 1.28 People with early onset type 2 diabetes
  Chunk ID : chunk_0382
  Score    : 0.5395
  Text     : the default assumption is that services will use the medicine with the lowest acquisiti

In [55]:
print("===== DAY 1 DELIVERABLE CHECKLIST =====\n")
print("1. PDFs loaded          :", PDF_PATHS)
print("2. Number of pages      :", len(all_pages))
print("3. Cleaned text sample  : see cleaning cell above")
print("4. Number of chunks     :", len(chunks))
print("5. Sample chunks        : see 3-chunk printout above")
print("6. Chroma DB created    : collection =", vector_db._collection.name,
      "| stored chunks =", vector_db._collection.count())
print("7. Test questions       : loaded live from RAG_Test_Questions.pdf —",
      len(supported_questions), "supported +", len(unsupported_questions), "unsupported")
print("8. Retrieval results    : Top-4 chunks + similarity scores printed for all",
      len(supported_questions) + len(unsupported_questions), "questions above")
print(f"\nSupported-question accuracy: {supported_hits}/{len(supported_questions)} "
      f"retrieved the correct source PDF + page (±3)")

===== DAY 1 DELIVERABLE CHECKLIST =====

1. PDFs loaded          : ['type-1-diabetes-in-adults-diagnosis-and-management-pdf-1837276469701.pdf', 'type-2-diabetes-in-adults-management-pdf-1837338615493.pdf']
2. Number of pages      : 195
3. Cleaned text sample  : see cleaning cell above
4. Number of chunks     : 462
5. Sample chunks        : see 3-chunk printout above
6. Chroma DB created    : collection = diabetes_educational_rag | stored chunks = 462
7. Test questions       : loaded live from RAG_Test_Questions.pdf — 15 supported + 5 unsupported
8. Retrieval results    : Top-4 chunks + similarity scores printed for all 20 questions above

Supported-question accuracy: 15/15 retrieved the correct source PDF + page (±3)
